**AutoCost AI – Multi-Agent Enterprise Cost Intelligence & Autonomous Action System**

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Upload Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'paysim1' dataset.
Path to dataset files: /kaggle/input/paysim1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(file_path)

print(df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [ ]:
import os
files_in_directory = os.listdir(path)
print(files_in_directory)

['PS_20174392719_1491204439457_log.csv']


In [ ]:
csv_file_name = files_in_directory[0]
full_file_path = os.path.join(path, csv_file_name)
df = pd.read_csv(full_file_path)
print("CSV file loaded successfully into a DataFrame.")
df.head()

CSV file loaded successfully into a DataFrame.


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
df.tail()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,0
6362619,743,CASH_OUT,850002.52,C1280323807,850002.52,0.0,C873221189,6510099.11,7360101.63,1,0


In [ ]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [ ]:
df.shape

(6362620, 11)

In [ ]:
import pandas as pd

def load_data(file):
  df = pd.read_csv(file)
  return df

In [ ]:
def detect_fraud(df):
    fraud_df = df[df["isFraud"] == 1]
    fraud_loss = fraud_df["amount"].sum()
    return fraud_df, fraud_loss

In [ ]:
def detect_duplicates(df):
    duplicate_df = df[df.duplicated(subset=["nameOrig","nameDest","amount"], keep=False)]
    duplicate_loss = duplicate_df["amount"].sum()
    return duplicate_df, duplicate_loss

In [ ]:
def detect_anomalies(df):
    threshold = df["amount"].mean() + 2 * df["amount"].std()
    anomaly_df = df[df["amount"] > threshold]
    anomaly_loss = anomaly_df["amount"].sum()
    return anomaly_df, anomaly_loss

In [ ]:
def calculate_total_savings(fraud_loss, duplicate_loss, anomaly_loss):
    total = fraud_loss + duplicate_loss + anomaly_loss
    yearly = total * 12
    return total, yearly

In [ ]:
def generate_report(fraud_loss, duplicate_loss, anomaly_loss, total, yearly):
    report = f"""
    AutoCost AI Report

    Fraud Loss Prevented: Rs {fraud_loss}
    Duplicate Payments Prevented: Rs {duplicate_loss}
    Anomaly Loss Prevented: Rs {anomaly_loss}

    Total Monthly Savings: Rs {total}
    Estimated Yearly Savings: Rs {yearly}
    """

    with open("report.txt", "w") as file:
        file.write(report)

    return report

In [ ]:
import matplotlib.pyplot as plt

def plot_savings(fraud, duplicate, anomaly):
    labels = ["Fraud", "Duplicate", "Anomaly"]
    values = [fraud, duplicate, anomaly]

    plt.figure()
    plt.bar(labels, values)
    plt.title("Cost Savings Breakdown")
    plt.xlabel("Category")
    plt.ylabel("Amount Saved")
    plt.show()

In [ ]:
!pip install streamlit

In [ ]:
!streamlit run app.py --server.maxUploadSize 1024 --server.port 8501 --server.address 0.0.0.0 &



2026-03-29 13:42:29.171 Port 8501 is not available


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="AutoCost AI", layout="wide")

st.title("AutoCost AI – Enterprise Cost Intelligence & Autonomous Action System")
st.write("Upload enterprise transaction dataset to detect cost leakage, fraud, and inefficiencies.")

# -------------------------------
# Load Dataset from Google Drive
# -------------------------------
st.sidebar.header("Load Dataset from Google Drive")

file_path = st.sidebar.text_input(
    "Enter Google Drive File Path",
    "/content/drive/MyDrive/PS_20174392719_1491204439457_log.csv"
)

if st.sidebar.button("Load Dataset"):
    df = pd.concat(pd.read_csv(file_path, chunksize=100000))
    st.session_state["df"] = df
    st.success("Dataset Loaded Successfully")

# -------------------------------
# If dataset loaded, show preview
# -------------------------------
if "df" in st.session_state:
    df = st.session_state["df"]

    st.subheader("Dataset Preview")
    st.write(df.head())

    # -------------------------------
    # Run Analysis
    # -------------------------------
    if st.button("Run AI Cost Analysis"):

        fraud_df = df[df["isFraud"] == 1]
        fraud_loss = fraud_df["amount"].sum()

        duplicate_df = df[df.duplicated(subset=["nameOrig","nameDest","amount"], keep=False)]
        duplicate_loss = duplicate_df["amount"].sum()

        threshold = df["amount"].mean() + 2 * df["amount"].std()
        anomaly_df = df[df["amount"] > threshold]
        anomaly_loss = anomaly_df["amount"].sum()

        total_savings = fraud_loss + duplicate_loss + anomaly_loss
        yearly_savings = total_savings * 12

        st.subheader("Detection Results")

        col1, col2, col3 = st.columns(3)
        col1.metric("Fraud Loss Prevented", f"Rs {fraud_loss:,}")
        col2.metric("Duplicate Payments Prevented", f"Rs {duplicate_loss:,}")
        col3.metric("Anomaly Loss Prevented", f"Rs {anomaly_loss:,}")

        st.subheader("Total Financial Impact")
        st.success(f"Total Monthly Savings: Rs {total_savings:,}")
        st.success(f"Estimated Yearly Savings: Rs {yearly_savings:,}")

        st.subheader("Cost Savings Breakdown")

        labels = ["Fraud", "Duplicate", "Anomaly"]
        values = [fraud_loss, duplicate_loss, anomaly_loss]

        fig = plt.figure()
        plt.bar(labels, values)
        plt.xlabel("Cost Leakage Type")
        plt.ylabel("Amount Saved")
        plt.title("Cost Savings Breakdown")
        st.pyplot(fig)

        report = f"""
AutoCost AI Report

Fraud Loss Prevented: Rs {fraud_loss}
Duplicate Payments Prevented: Rs {duplicate_loss}
Anomaly Loss Prevented: Rs {anomaly_loss}

Total Monthly Savings: Rs {total_savings}
Estimated Yearly Savings: Rs {yearly_savings}
"""

        st.subheader("Generated Action Report")
        st.text(report)

        st.download_button("Download Report", report, file_name="AutoCost_Report.txt")

Overwriting app.py


In [ ]:
!pip install streamlit pyngrok pandas matplotlib

In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Kill existing tunnels
ngrok.kill()

# Run Streamlit
process = subprocess.Popen(["streamlit", "run", "app.py"])

# Wait for Streamlit to start
time.sleep(5)

# Create ngrok tunnel on a different port
public_url = ngrok.connect(8501) # Changed port from 8501 to 8502
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://nonreligious-maximiliano-overcrowdedly.ngrok-free.dev" -> "http://localhost:8501"
